# 08. Prompt Experiment Design

Good prompt evaluation is not just about scoring outputs. It is also about designing experiments that produce trustworthy conclusions. This notebook focuses on ablations, temperature controls, sample size, and ways misleading wins sneak into prompt work.

## Learning goals

By the end you will be able to:

- design a prompt ablation plan instead of random prompt tweaking
- control for temperature and sampling variance
- reason about sample size with simple simulations
- build experiment tables and short written summaries
- explain why early wins can be misleading

## Notebook setup

This notebook stays lightweight and pure Python. We will simulate prompt experiments so the logic stays transparent and reproducible.

In [ ]:
import math
import random
import statistics
from IPython.display import Markdown, display

random.seed(23)

def show_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    display(Markdown("\n".join([header, divider] + body)))

def mean(values):
    values = list(values)
    return round(sum(values) / len(values), 3) if values else 0.0

## Step 1: Create a benchmark with slices

Experiment design improves when you know what kinds of tasks you are testing. We will create a small benchmark with slices such as extraction, reasoning, rewriting, and classification.

In [ ]:
TASK_SLICES = ["extraction", "reasoning", "rewriting", "classification"]

def make_tasks(n=72, seed=0):
    rng = random.Random(seed)
    tasks = []
    for index in range(n):
        task_slice = TASK_SLICES[index % len(TASK_SLICES)]
        tasks.append(
            {
                "id": f"task_{index:02d}",
                "slice": task_slice,
                "difficulty": round(rng.uniform(0.15, 0.95), 2),
                "format_strictness": round(rng.uniform(0.1, 0.9), 2),
            }
        )
    return tasks

tasks = make_tasks()
show_table(tasks[:8], columns=["id", "slice", "difficulty", "format_strictness"])

## Step 2: Define prompt ablations

Ablations should isolate one idea at a time. Here we compare a baseline prompt, constraints only, few-shot examples only, and the combination.

In [ ]:
VARIANTS = {
    "baseline": {
        "quality_bonus": 0.00,
        "format_bonus": 0.00,
        "cost_tokens": 110,
        "slice_bonus": {},
    },
    "constraints_only": {
        "quality_bonus": 0.03,
        "format_bonus": 0.10,
        "cost_tokens": 150,
        "slice_bonus": {"extraction": 0.05, "classification": 0.03},
    },
    "fewshot_only": {
        "quality_bonus": 0.06,
        "format_bonus": 0.03,
        "cost_tokens": 210,
        "slice_bonus": {"reasoning": 0.07, "rewriting": 0.03},
    },
    "combo": {
        "quality_bonus": 0.09,
        "format_bonus": 0.11,
        "cost_tokens": 260,
        "slice_bonus": {"extraction": 0.05, "reasoning": 0.07, "classification": 0.03},
    },
}

show_table(
    [
        {
            "variant": name,
            "quality_bonus": settings["quality_bonus"],
            "format_bonus": settings["format_bonus"],
            "cost_tokens": settings["cost_tokens"],
        }
        for name, settings in VARIANTS.items()
    ]
)

## Step 3: Simulate model behavior

The simulator gives every task a latent probability of success. Variant design, task difficulty, formatting pressure, and temperature all influence the observed result.

In [ ]:
def clamp(value, low=0.01, high=0.99):
    return max(low, min(high, value))

def success_probability(task, variant_name, temperature):
    settings = VARIANTS[variant_name]
    base = 0.78 - 0.48 * task["difficulty"]
    slice_bonus = settings["slice_bonus"].get(task["slice"], 0.0)
    temperature_penalty = 0.08 * temperature
    return clamp(base + settings["quality_bonus"] + slice_bonus - temperature_penalty)

def format_probability(task, variant_name, temperature):
    settings = VARIANTS[variant_name]
    base = 0.82 - 0.35 * task["format_strictness"]
    temperature_penalty = 0.12 * temperature
    return clamp(base + settings["format_bonus"] - temperature_penalty)

def run_experiment(sampled_tasks, variant_name, temperature=0.2, seed=0):
    rng = random.Random(seed)
    scored = []
    for task in sampled_tasks:
        quality = 1 if rng.random() < success_probability(task, variant_name, temperature) else 0
        format_ok = 1 if rng.random() < format_probability(task, variant_name, temperature) else 0
        final_score = 1 if quality and format_ok else 0
        scored.append(
            {
                "id": task["id"],
                "slice": task["slice"],
                "quality": quality,
                "format_ok": format_ok,
                "final_score": final_score,
            }
        )
    return scored

## Step 4: Run a first ablation table

A first table should report more than one metric. Here we track task success, formatting reliability, and a simple token-cost proxy.

In [ ]:
def summarize_run(sampled_tasks, variant_name, temperature=0.2, seed=0):
    rows = run_experiment(sampled_tasks, variant_name, temperature=temperature, seed=seed)
    return {
        "variant": variant_name,
        "temperature": temperature,
        "n_tasks": len(rows),
        "success_rate": mean(row["final_score"] for row in rows),
        "quality_rate": mean(row["quality"] for row in rows),
        "format_rate": mean(row["format_ok"] for row in rows),
        "cost_tokens": VARIANTS[variant_name]["cost_tokens"],
    }

pilot_tasks = tasks[:24]
pilot_summary = [summarize_run(pilot_tasks, name, temperature=0.2, seed=5) for name in VARIANTS]
show_table(pilot_summary, columns=["variant", "n_tasks", "success_rate", "quality_rate", "format_rate", "cost_tokens"])

## Step 5: Control temperature before comparing prompts

If prompt A runs at temperature 0.0 and prompt B runs at 0.7, the experiment is confounded. Temperature changes variance and formatting reliability, so we should measure it explicitly.

In [ ]:
temperature_rows = []
for temperature in [0.0, 0.2, 0.5, 0.8]:
    summary = summarize_run(pilot_tasks, "combo", temperature=temperature, seed=8)
    temperature_rows.append(summary)

show_table(temperature_rows, columns=["variant", "temperature", "success_rate", "quality_rate", "format_rate", "cost_tokens"])

## Step 6: Compare full-dataset results

Small pilots are helpful, but you should also inspect the more stable picture on the full benchmark.

In [ ]:
full_summary = [summarize_run(tasks, name, temperature=0.2, seed=9) for name in VARIANTS]
show_table(full_summary, columns=["variant", "n_tasks", "success_rate", "quality_rate", "format_rate", "cost_tokens"])

## Step 7: Bootstrap a lift interval

Raw lift can be misleading without uncertainty. A lightweight bootstrap gives a rough interval for the observed difference between two prompt variants.

In [ ]:
def bootstrap_lift(sampled_tasks, variant_a, variant_b, temperature=0.2, trials=1000, seed=0):
    rng = random.Random(seed)
    lifts = []
    for trial in range(trials):
        resampled = [rng.choice(sampled_tasks) for _ in sampled_tasks]
        score_a = summarize_run(resampled, variant_a, temperature=temperature, seed=trial)["success_rate"]
        score_b = summarize_run(resampled, variant_b, temperature=temperature, seed=trial + 10000)["success_rate"]
        lifts.append(score_b - score_a)
    lifts.sort()
    low = lifts[int(0.05 * len(lifts))]
    high = lifts[int(0.95 * len(lifts)) - 1]
    return round(low, 3), round(high, 3)

low, high = bootstrap_lift(tasks, "baseline", "combo", temperature=0.2, seed=12)
print({"comparison": "combo minus baseline", "bootstrap_90pct_ci": (low, high)})

## Step 8: Study sample size

One of the easiest ways to fool yourself is to compare prompts on too few examples. We will repeat the same experiment many times at different sample sizes and see how stable the winner is.

In [ ]:
def sample_size_stability(n_tasks, repeats=300, temperature=0.2, seed=0):
    rng = random.Random(seed)
    lifts = []
    combo_wins = 0
    for repeat in range(repeats):
        sampled = rng.sample(tasks, n_tasks)
        baseline_score = summarize_run(sampled, "baseline", temperature=temperature, seed=repeat)["success_rate"]
        combo_score = summarize_run(sampled, "combo", temperature=temperature, seed=repeat + 5000)["success_rate"]
        lift = combo_score - baseline_score
        lifts.append(lift)
        if lift > 0:
            combo_wins += 1
    return {
        "n_tasks": n_tasks,
        "mean_lift": round(statistics.mean(lifts), 3),
        "stdev_lift": round(statistics.pstdev(lifts), 3),
        "combo_win_frequency": round(combo_wins / repeats, 3),
    }

size_rows = [sample_size_stability(size, seed=17) for size in [8, 16, 32, 64]]
show_table(size_rows)

## Step 9: Demonstrate a misleading win

Early stopping can make noise look like signal. In the next cell we compare two equally strong prompts and allow peeking every few tasks until one appears to lead.

In [ ]:
VARIANTS["baseline_clone"] = {
    "quality_bonus": VARIANTS["baseline"]["quality_bonus"],
    "format_bonus": VARIANTS["baseline"]["format_bonus"],
    "cost_tokens": VARIANTS["baseline"]["cost_tokens"],
    "slice_bonus": dict(VARIANTS["baseline"]["slice_bonus"]),
}

def peeking_false_win_rate(max_tasks=40, step=5, repeats=300, temperature=0.2, seed=0):
    rng = random.Random(seed)
    false_wins = 0
    for repeat in range(repeats):
        sampled = rng.sample(tasks, max_tasks)
        declared_winner = None
        for current_n in range(step, max_tasks + 1, step):
            current_tasks = sampled[:current_n]
            score_a = summarize_run(current_tasks, "baseline", temperature=temperature, seed=repeat)["success_rate"]
            score_b = summarize_run(current_tasks, "baseline_clone", temperature=temperature, seed=repeat + 9000)["success_rate"]
            if abs(score_a - score_b) >= 0.10:
                declared_winner = "baseline" if score_a > score_b else "baseline_clone"
                break
        if declared_winner is not None:
            false_wins += 1
    return round(false_wins / repeats, 3)

print({"false_win_rate_with_peeking": peeking_false_win_rate(seed=21)})

## Step 10: Slice the benchmark

Aggregate results can hide where a prompt actually helps. Stratified summaries show whether a win is broad or concentrated in one task family.

In [ ]:
def summarize_by_slice(sampled_tasks, variant_name, temperature=0.2, seed=0):
    rows = run_experiment(sampled_tasks, variant_name, temperature=temperature, seed=seed)
    grouped = {}
    for row in rows:
        grouped.setdefault(row["slice"], []).append(row["final_score"])
    return {
        task_slice: round(statistics.mean(scores), 3)
        for task_slice, scores in sorted(grouped.items())
    }

slice_rows = []
for variant_name in ["baseline", "constraints_only", "fewshot_only", "combo"]:
    row = {"variant": variant_name}
    row.update(summarize_by_slice(tasks, variant_name, temperature=0.2, seed=27))
    slice_rows.append(row)

show_table(slice_rows, columns=["variant"] + TASK_SLICES)

## Step 11: Build an experiment table

A good experiment table should combine setup, results, and interpretation. This makes later comparisons easier to audit.

In [ ]:
def experiment_report_rows(sampled_tasks, temperature, seed):
    rows = []
    for variant_name in ["baseline", "constraints_only", "fewshot_only", "combo"]:
        summary = summarize_run(sampled_tasks, variant_name, temperature=temperature, seed=seed)
        rows.append(
            {
                "variant": variant_name,
                "temp": temperature,
                "n": summary["n_tasks"],
                "success": summary["success_rate"],
                "format": summary["format_rate"],
                "cost_tokens": summary["cost_tokens"],
                "notes": "best raw quality" if variant_name == "combo" else "cheaper" if variant_name == "baseline" else "middle ground",
            }
        )
    return rows

report_rows = experiment_report_rows(tasks, temperature=0.2, seed=31)
show_table(report_rows, columns=["variant", "temp", "n", "success", "format", "cost_tokens", "notes"])

## Step 12: Write a compact summary

Numbers are not the whole deliverable. Teams usually need a short written summary that explains what changed, how strong the evidence is, and what to do next.

In [ ]:
def summarize_family(rows):
    best = max(rows, key=lambda row: row["success"])
    cheapest = min(rows, key=lambda row: row["cost_tokens"])
    return {
        "best_variant": best["variant"],
        "best_success_rate": best["success"],
        "cheapest_variant": cheapest["variant"],
        "cheapest_cost_tokens": cheapest["cost_tokens"],
        "recommended_next_step": "Run a larger confirmatory experiment on combo vs fewshot_only with fixed temperature and pre-registered slices.",
    }

summary_note = summarize_family(report_rows)
summary_note

## Design principles to keep

Strong prompt experiments usually:

- isolate one prompt change at a time
- keep model, temperature, and dataset fixed across variants
- report uncertainty, not just point estimates
- inspect task slices instead of trusting one aggregate
- avoid peeking and stopping the moment a tiny lead appears

## Suggested extensions

Try these next:

1. add latency and dollar-cost proxies to the experiment table
2. simulate a judge-based metric instead of a binary success metric
3. compare confirmatory runs against exploratory pilot runs